# 01 — EDA, перевірка розмітки та розбиття на фолди

Спільна частина (розділи 0–5) потрібна обом учасникам, частина Person A (розділи 6–7) готує дані для torchvision-моделей. Ноутбук працює на CPU.

## 0. Налаштування
Імпортуємо бібліотеки, за потреби встановлюємо `pycocotools` і визначаємо папки для результатів та графіків для звіту.

In [ ]:
import os, sys, json, math, time, shutil, subprocess, importlib, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
import torch
try:
    import pycocotools
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pycocotools"], check=True)

ON_KAGGLE = Path("/kaggle/working").exists()
WORK = Path("/kaggle/working") if ON_KAGGLE else Path.cwd()
IMG_OUT = WORK / "images" if ON_KAGGLE else Path.cwd().parent / "images"
IMG_OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 100, "savefig.bbox": "tight", "savefig.dpi": 130})
print("Kaggle:", ON_KAGGLE, "| work dir:", WORK, "| plots:", IMG_OUT)

Підтягуємо спільні модулі `common.py` і `tv_dataset.py`: з робочої папки, з доданого до ноутбука Kaggle Dataset або з нашого GitHub-репозиторію, і фіксуємо seed.

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/KaplyshOlha/hw-2/main/notebooks"
for name in ["common.py", "tv_dataset.py"]:
    if (WORK / name).exists():
        continue
    found = [p for pat in ["*", "*/*", "*/*/*"] for p in sorted(Path("/kaggle/input").glob(f"{pat}/{name}"))]
    if found:
        shutil.copy(found[0], WORK / name)
    else:
        urllib.request.urlretrieve(f"{REPO_RAW}/{name}", WORK / name)
sys.path.insert(0, str(WORK))
import common, tv_dataset
importlib.reload(common); importlib.reload(tv_dataset)
common.seed_everything(common.SEED)
rng = np.random.default_rng(common.SEED)
print(common.__file__)

## 1. Структура датасету
Знаходимо корінь даних (перевіряємо шлях з інструкцій) і показуємо вміст верхнього рівня.

In [ ]:
EXPECTED = Path(f"/kaggle/input/{common.SLUG}")
ROOT = common.find_data_root()
print("expected path exists:", EXPECTED.exists(), "| using:", ROOT)
for p in sorted(ROOT.iterdir()):
    print(f"  {p.name:<24}", "dir" if p.is_dir() else f"{p.stat().st_size / 1e6:.1f} MB")

Рахуємо кліпи, кадри, файли міток і файли ігнорованих регіонів у кожному спліті та порівнюємо з числами з опису змагання.

In [ ]:
expected = {"train": (56, 24201), "val": (7, 2846), "test": (48, 3811)}
rows = []
for split, (exp_clips, exp_frames) in expected.items():
    d = ROOT / split
    clips = sorted(c.name for c in (d / "images").iterdir() if c.is_dir())
    count = lambda sub: sum(len(os.listdir(d / sub / c)) for c in clips if (d / sub / c).exists())
    rows.append(dict(split=split, clips=len(clips), exp_clips=exp_clips, frames=count("images"),
                     exp_frames=exp_frames, label_files=count("labels"), ignore_files=count("ignore")))
pd.DataFrame(rows).set_index("split")

Переглядаємо `data.yaml`, категорії COCO-анотацій, перші рядки `test_images.csv` і `sample_submission.csv` та перевіряємо, що їхні `image_id` збігаються.

In [ ]:
print((ROOT / "train" / "data.yaml").read_text())
with open(ROOT / "val" / "annotations_coco.json") as f:
    print("COCO categories:", [(c["id"], c["name"]) for c in json.load(f)["categories"]])
test_index = common.load_test_index(ROOT)
sample_sub = pd.read_csv(ROOT / "sample_submission.csv")
print("test rows:", len(test_index), "| sample rows:", len(sample_sub),
      "| same ids:", test_index.image_id.tolist() == sample_sub.image_id.tolist())
display(test_index.head(3)); display(sample_sub.head(3))

## 2. Завантаження та перевірка розмітки
Зчитуємо COCO-анотації train і val в одну таблицю кадрів (`images`) і одну таблицю боксів (`anns`) з глобальним `img_id`; результат кешується в `gt_cache.pkl`.

In [ ]:
t0 = time.time()
images, anns = common.load_gt(ROOT, cache=WORK / "gt_cache.pkl")
obj = (anns[anns.iscrowd == 0]
       .merge(images[["img_id", "split", "clip", "width", "height"]], on="img_id")
       .sort_values("img_id", kind="stable").reset_index(drop=True))
ign = anns[anns.iscrowd == 1].reset_index(drop=True)
obj["cname"] = obj.cls.map(dict(enumerate(common.CLASSES)))
print(f"{time.time() - t0:.0f}s | frames {len(images)} | clips {images['clip'].nunique()} | "
      f"boxes {len(obj)} | ignored regions {len(ign)}")
images.head(3)

Перевіряємо, як оригінальні класи VisDrone (`orig_category_id`) відображені на наші 8 класів і що потрапляє в ігноровані регіони (non-target класи).

In [ ]:
pd.crosstab(anns.orig_cat, anns.cls.map(dict(enumerate(common.CLASSES))).fillna("ignored (iscrowd=1)"))

Звіряємо YOLO-мітки (`labels/`) і файли `ignore/` з COCO-анотаціями на 30 випадкових кадрах: кількість боксів і максимальну різницю координат у пікселях.

In [ ]:
def yolo_to_xywh(path, w, h, has_cls):
    if not path.exists() or path.stat().st_size == 0:
        return np.zeros((0, 5 if has_cls else 4))
    a = np.loadtxt(path, ndmin=2)
    b = a[:, 1:] if has_cls else a
    xywh = np.column_stack([(b[:, 0] - b[:, 2] / 2) * w, (b[:, 1] - b[:, 3] / 2) * h, b[:, 2] * w, b[:, 3] * h])
    return np.column_stack([a[:, 0], xywh]) if has_cls else xywh

def max_diff(a, b):
    if len(a) != len(b):
        return np.inf
    if len(a) == 0:
        return 0.0
    sort = lambda m: m[np.lexsort(np.round(m).T[::-1])]
    return float(np.abs(sort(a) - sort(b)).max())

checks = []
for iid in rng.choice(images.img_id.to_numpy(), 30, replace=False):
    r = images.loc[iid]
    d, stem = ROOT / r.split, Path(r.path).stem
    lab = yolo_to_xywh(d / "labels" / r["clip"] / f"{stem}.txt", r.width, r.height, True)
    ig = yolo_to_xywh(d / "ignore" / r["clip"] / f"{stem}.txt", r.width, r.height, False)
    o, g = obj[obj.img_id == iid], ign[ign.img_id == iid]
    checks.append(dict(n_yolo=len(lab), n_coco=len(o), n_ign_txt=len(ig), n_ign_coco=len(g),
                       diff_boxes=max_diff(lab, o[["cls", "x", "y", "w", "h"]].to_numpy(float)),
                       diff_ign=max_diff(ig, g[["x", "y", "w", "h"]].to_numpy(float))))
checks = pd.DataFrame(checks)
print("max coordinate diff (px):", checks[["diff_boxes", "diff_ign"]].max().round(3).to_dict(),
      "| frames with count mismatch:",
      int(((checks.n_yolo != checks.n_coco) | (checks.n_ign_txt != checks.n_ign_coco)).sum()), "/ 30")

## 3. EDA
### 3.1 Розподіл класів
Рахуємо кількість боксів кожного класу загалом і в оригінальних train/val та будуємо діаграму в лог-шкалі.

In [ ]:
cls_counts = pd.crosstab(obj.cname, obj.split).reindex(common.CLASSES).fillna(0).astype(int)
cls_counts["total"] = cls_counts.sum(axis=1)
cls_counts["share_%"] = (100 * cls_counts.total / cls_counts.total.sum()).round(2)
fig, ax = plt.subplots(figsize=(8, 3.2))
cls_counts.total.plot.bar(ax=ax, logy=True, color="tab:blue", rot=0)
ax.set_ylabel("boxes (log)"); ax.set_xlabel(""); ax.set_title("Class distribution, train + val")
fig.savefig(IMG_OUT / "class_distribution.png"); plt.show()
cls_counts

Дивимось, у скількох кліпах трапляється кожен клас і як частки класів змінюються між кліпами (теплова карта, кліпи відсортовані за часткою `car`).

In [ ]:
cnt = pd.crosstab(obj["clip"], obj.cname).reindex(columns=common.CLASSES, fill_value=0)
share = cnt.div(cnt.sum(axis=1), axis=0)
print("clips containing class:", (cnt > 0).sum().to_dict())
fig, ax = plt.subplots(figsize=(12, 3.2))
im = ax.imshow(share.sort_values("car").T.values, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(common.CLASSES)), common.CLASSES); ax.set_xticks([])
ax.set_xlabel(f"{len(share)} clips, sorted by car share"); fig.colorbar(im, ax=ax, label="share of boxes")
fig.savefig(IMG_OUT / "class_share_per_clip.png"); plt.show()

### 3.2 Розміри об'єктів
Обчислюємо √площі боксу в оригінальних пікселях і частки COCO small (<32²) / medium / large (≥96²) загалом та по класах.

In [ ]:
obj["area"] = obj.w * obj.h
obj["sqrt_area"] = np.sqrt(obj.area.astype(np.float64))
obj["size_cat"] = pd.cut(obj.area, [0, 32 ** 2, 96 ** 2, np.inf], right=False,
                         labels=["small", "medium", "large"]).astype(str)
size_tab = (pd.crosstab(obj.cname, obj.size_cat, normalize="index").mul(100)
            .reindex(index=common.CLASSES, columns=["small", "medium", "large"]).fillna(0))
size_tab.loc["ALL"] = obj.size_cat.value_counts(normalize=True).reindex(["small", "medium", "large"]).fillna(0).values * 100
size_tab["median_sqrt_area_px"] = obj.groupby("cname").sqrt_area.median().reindex(size_tab.index)
size_tab.loc["ALL", "median_sqrt_area_px"] = obj.sqrt_area.median()
size_tab.round(1)

Будуємо гістограми √площі по класах і порівнюємо розмір об'єктів в оригіналі та після масштабування кадру до найдовшої сторони 640 і 1280 px (так їх побачить модель).

In [ ]:
long_side = np.maximum(obj.width, obj.height)
obj["sqrt_area_640"] = obj.sqrt_area * 640 / long_side
bins = np.logspace(0, np.log10(obj.sqrt_area.max() + 1), 60)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for c in common.CLASSES:
    axes[0].hist(obj.sqrt_area[obj.cname == c], bins=bins, histtype="step", density=True, label=c)
axes[0].set_title("sqrt(area) by class, original px"); axes[0].legend(fontsize=7, ncol=2)
axes[1].hist([obj.sqrt_area, obj.sqrt_area * 1280 / long_side, obj.sqrt_area_640], bins=bins,
             histtype="step", density=True, label=["original", "long side 1280", "long side 640"])
axes[1].set_title("sqrt(area) after resizing")
for ax, lines in [(axes[0], [32, 96]), (axes[1], [8, 16, 32])]:
    ax.set_xscale("log")
    for v in lines:
        ax.axvline(v, ls="--", c="gray", lw=0.8)
axes[1].legend(fontsize=8)
fig.savefig(IMG_OUT / "object_sizes.png"); plt.show()
q = [0.05, 0.25, 0.5, 0.75, 0.95]
size_q = pd.DataFrame({"original": obj.sqrt_area.quantile(q), "long side 1280": (obj.sqrt_area * 1280 / long_side).quantile(q),
                       "long side 640": obj.sqrt_area_640.quantile(q)}).T
size_q["<8px %"] = [100 * (s < 8).mean() for s in [obj.sqrt_area, obj.sqrt_area * 1280 / long_side, obj.sqrt_area_640]]
size_q.round(1)

Порівнюємо роздільні здатності кадрів у train / val / test і для кожного кліпу рахуємо кількість кадрів, боксів, частку малих об'єктів та медіанний розмір.

In [ ]:
res = pd.concat([images.groupby("clip").first()[["split", "width", "height"]],
                 test_index.groupby("clip").first()[["width", "height"]].assign(split="test")])
res["resolution"] = res.width.astype(str) + "x" + res.height.astype(str)
display(pd.crosstab(res.resolution, res.split, margins=True))

g = obj.groupby("clip")
clip_stats = images.groupby("clip").agg(split=("split", "first"), frames=("img_id", "size"),
                                        width=("width", "first"), height=("height", "first"))
clip_stats["boxes"] = g.size().reindex(clip_stats.index, fill_value=0)
clip_stats["small_%"] = (obj.size_cat == "small").groupby(obj["clip"]).mean().mul(100)
clip_stats["median_sqrt_area"] = g.sqrt_area.median()
clip_stats["top_class"] = g.cname.agg(lambda s: s.value_counts().index[0])

fig, axes = plt.subplots(1, 2, figsize=(13, 3.6))
for sp, c in [("train", "tab:blue"), ("val", "tab:orange")]:
    cs = clip_stats[clip_stats.split == sp]
    axes[0].scatter(cs.width, cs.median_sqrt_area, c=c, label=sp, alpha=0.7)
axes[0].set_xlabel("frame width, px"); axes[0].set_ylabel("median sqrt(area), px"); axes[0].legend()
cs = clip_stats.sort_values("small_%")
axes[1].bar(range(len(cs)), cs["small_%"], color=np.where(cs.split == "val", "tab:orange", "tab:blue"))
axes[1].set_xlabel("clip (sorted)"); axes[1].set_ylabel("COCO small, %"); axes[1].set_xticks([])
fig.savefig(IMG_OUT / "sizes_per_clip.png"); plt.show()
clip_stats[["frames", "boxes", "small_%", "median_sqrt_area"]].describe().round(1)

### 3.3 Щільність сцен
Рахуємо кількість об'єктів на кадр (включно з порожніми кадрами) і показуємо розподіл загалом та по кліпах; вертикальні лінії — ліміти детекцій 100 / 300 / 500.

In [ ]:
images["n_obj"] = obj.groupby("img_id").size().reindex(images.index, fill_value=0)
print(images.n_obj.describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_dict())
print({f">{k}": int((images.n_obj > k).sum()) for k in [100, 300, 500]}, "frames with no objects:", int((images.n_obj == 0).sum()))
fig, axes = plt.subplots(1, 2, figsize=(13, 3.6), gridspec_kw={"width_ratios": [1, 2.2]})
axes[0].hist(images.n_obj, bins=60, color="tab:blue")
for v, c in [(100, "tab:red"), (300, "tab:orange"), (500, "k")]:
    axes[0].axvline(v, ls="--", c=c, lw=0.8, label=f"{v}")
axes[0].set_xlabel("objects per frame"); axes[0].legend(fontsize=8)
order = images.groupby("clip").n_obj.median().sort_values().index
axes[1].boxplot([images.n_obj[images["clip"] == c] for c in order], showfliers=False)
axes[1].set_xticks([]); axes[1].set_xlabel("clip (sorted by median)"); axes[1].set_ylabel("objects per frame")
fig.savefig(IMG_OUT / "objects_per_frame.png"); plt.show()
clip_stats["obj_per_frame"] = images.groupby("clip").n_obj.mean()

### 3.4 Ігноровані регіони
Для кожного кадру будуємо маску ігнорованих регіонів, рахуємо частку площі кадру під нею і для кожного боксу — частку його площі всередині ігнору.

In [ ]:
oid = obj.img_id.to_numpy()
oxywh = obj[["x", "y", "w", "h"]].to_numpy(np.float64)
cover = np.zeros(len(obj), np.float32)
frac = {}
t0 = time.time()
for iid, g in ign.groupby("img_id"):
    m = common.ignore_mask(images.at[iid, "height"], images.at[iid, "width"], g[["x", "y", "w", "h"]].to_numpy())
    frac[iid] = m.mean()
    a, b = np.searchsorted(oid, [iid, iid + 1])
    if b > a:
        cover[a:b] = common.box_coverage(oxywh[a:b], m)
obj["ign_cover"] = cover
images["ign_frac"] = pd.Series(frac, dtype=float).reindex(images.index, fill_value=0.0)
images["n_ign"] = ign.groupby("img_id").size().reindex(images.index, fill_value=0)
has = images.n_ign > 0
print(f"{time.time() - t0:.0f}s | frames with ignore: {100 * has.mean():.1f}% | "
      f"regions per such frame: {images.n_ign[has].mean():.1f} | covered area there: {100 * images.ign_frac[has].mean():.1f}% | "
      f"boxes >=50% inside ignore: {100 * (obj.ign_cover >= 0.5).mean():.2f}%")

Показуємо розподіл частки площі під ігнором і як вона змінюється між кліпами, а також по класах — скільки боксів перекрито ігнором на ≥50%.

In [ ]:
clip_stats["ignore_frames_%"] = has.groupby(images["clip"]).mean().mul(100)
clip_stats["ignore_area_%"] = images.groupby("clip").ign_frac.mean().mul(100)
fig, axes = plt.subplots(1, 2, figsize=(13, 3.4))
axes[0].hist(100 * images.ign_frac[has], bins=40); axes[0].set_xlabel("ignored area, % of frame (frames with ignore)")
cs = clip_stats.sort_values("ignore_area_%")
axes[1].bar(range(len(cs)), cs["ignore_area_%"]); axes[1].set_xticks([])
axes[1].set_xlabel("clip (sorted)"); axes[1].set_ylabel("mean ignored area, %")
fig.savefig(IMG_OUT / "ignore_regions.png"); plt.show()
(obj.ign_cover >= 0.5).groupby(obj.cname).mean().mul(100).reindex(common.CLASSES).round(2).rename("boxes >=50% in ignore, %").to_frame().T

### 3.5 Оклюзія та обрізання
Рахуємо частки рівнів оклюзії і обрізання (поля `occlusion`, `truncation`) для кожного класу.

In [ ]:
occ = pd.crosstab(obj.cname, obj.occlusion, normalize="index").mul(100).reindex(common.CLASSES)
trn = pd.crosstab(obj.cname, obj.truncation, normalize="index").mul(100).reindex(common.CLASSES)
pd.concat([occ, trn], axis=1, keys=["occlusion, %", "truncation, %"]).round(1)

### 3.6 Часова кореляція кадрів
Для різних кроків прорідження кадрів рахуємо, скільки кадрів і боксів залишається, яку частку унікальних треків (`track_id`) ми ще бачимо, і наскільки відрізняються кадри t та t+крок (середня абсолютна різниця пікселів).

In [ ]:
track_key = obj["clip"].astype(str) + "_" + obj.track_id.astype(str)
n_tracks = track_key.nunique()
sample_clips = rng.choice(images["clip"].unique(), min(8, images["clip"].nunique()), replace=False)
small = lambda p: cv2.resize(cv2.imread(p, cv2.IMREAD_GRAYSCALE), (160, 90)).astype(np.float32)
rows = []
for s in [1, 2, 5, 10, 20, 40, 80]:
    keep = common.subsample(images, s).index
    sel = obj.img_id.isin(keep)
    diffs = []
    for c in sample_clips:
        fr = images[images["clip"] == c].sort_values("frame_id")
        if len(fr) > s:
            diffs.append(np.abs(small(fr.path.iloc[0]) - small(fr.path.iloc[s])).mean())
    rows.append(dict(stride=s, frames=len(keep), boxes=int(sel.sum()),
                     tracks_seen_pct=100 * track_key[sel].nunique() / n_tracks,
                     pixel_diff_t_vs_t_plus_stride=np.mean(diffs) if diffs else np.nan))
stride_tab = pd.DataFrame(rows).set_index("stride")
track_len = track_key.value_counts()
print("track length, frames:", track_len.describe(percentiles=[0.25, 0.5, 0.75]).round(0).to_dict())
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
stride_tab.tracks_seen_pct.plot(ax=axes[0], marker="o", logx=True); axes[0].set_ylabel("unique tracks seen, %")
stride_tab.pixel_diff_t_vs_t_plus_stride.plot(ax=axes[1], marker="o", logx=True); axes[1].set_ylabel("mean |frame t - frame t+s|")
for ax in axes:
    ax.axvline(common.TRAIN_STRIDE, ls="--", c="gray", lw=0.8)
fig.savefig(IMG_OUT / "stride_analysis.png"); plt.show()
stride_tab.round(1)

## 4. Візуальна перевірка розмітки
Малюємо GT-бокси (колір = клас) та ігноровані регіони (сіра штриховка) на кадрах з різних кліпів: найщільніший кадр, кадр з найдрібнішими об'єктами, кадр з найбільшою площею ігнору і три випадкові.

In [ ]:
COLORS = plt.cm.tab10(np.arange(len(common.CLASSES)))
ocls = obj.cls.to_numpy()

def draw_frame(ax, iid, fill=False):
    r = images.loc[iid]
    img = cv2.cvtColor(cv2.imread(r.path), cv2.COLOR_BGR2RGB)
    g = ign.loc[ign.img_id == iid, ["x", "y", "w", "h"]].to_numpy()
    if fill and len(g):
        img = common.fill_ignore(img, common.ignore_mask(*img.shape[:2], g))
    ax.imshow(img)
    a, b = np.searchsorted(oid, [iid, iid + 1])
    for (x, y, w, h), c in zip(oxywh[a:b], ocls[a:b]):
        ax.add_patch(mpatches.Rectangle((x, y), w, h, fill=False, ec=COLORS[c], lw=0.7))
    for x, y, w, h in g:
        ax.add_patch(mpatches.Rectangle((x, y), w, h, fill=not fill, fc=(0.5, 0.5, 0.5, 0.3), ec="white", hatch="//", lw=0.7))
    ax.set_title(f"{r['clip']} #{r.frame_id} | {r.width}x{r.height} | {b - a} objects", fontsize=8)
    ax.axis("off")

med = obj.groupby("img_id").sqrt_area.median()
med = med[images.n_obj.reindex(med.index) >= 20]
pick = [images.n_obj.idxmax(), med.idxmin() if len(med) else images.index[0], images.ign_frac.idxmax()]
other = images[~images["clip"].isin(images.loc[pick, "clip"])]
pick += other.groupby("clip").sample(1, random_state=common.SEED).sample(3, random_state=common.SEED).index.tolist()
legend = [mpatches.Patch(color=COLORS[i], label=c) for i, c in enumerate(common.CLASSES)]
legend.append(mpatches.Patch(fc=(0.5, 0.5, 0.5, 0.3), ec="k", hatch="//", label="ignored"))
fig, axes = plt.subplots(2, 3, figsize=(18, 7.5))
for ax, iid in zip(axes.ravel(), pick):
    draw_frame(ax, iid)
fig.legend(handles=legend, loc="lower center", ncol=9, fontsize=9)
fig.savefig(IMG_OUT / "gt_samples.png"); plt.show()

Той самий щільний кадр до і після заливки ігнорованих регіонів сірим (114) — саме так кадр бачитиме модель під час навчання.

In [ ]:
cand = images[(images.ign_frac > 0.03) & (images.ign_frac < 0.4)]
iid = cand.n_obj.idxmax() if len(cand) else images.ign_frac.idxmax()
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
draw_frame(axes[0], iid); draw_frame(axes[1], iid, fill=True)
fig.savefig(IMG_OUT / "ignore_fill.png"); plt.show()

## 5. Розбиття на фолди і спільна оцінка
Будуємо фіксоване 3-fold GroupKFold розбиття за кліпами по всіх 63 відео (train + val разом), перевіряємо відсутність перетинів і зберігаємо в `splits.json`.

In [ ]:
splits = common.make_splits(images, n_folds=common.N_FOLDS)
common.save_splits(splits, WORK / "splits.json")
val_sets = [set(f["val_clips"]) for f in splits["folds"]]
assert set().union(*val_sets) == set(images["clip"]) and sum(map(len, val_sets)) == images["clip"].nunique()
assert all(not set(f["train_clips"]) & set(f["val_clips"]) for f in splits["folds"])
assert common.load_splits(WORK / "splits.json") == json.loads(json.dumps(splits))
print({f["fold"]: f"{len(f['train_clips'])} train / {len(f['val_clips'])} val clips" for f in splits["folds"]})

Перевіряємо баланс фолдів: кліпи, кадри (усього, train-кадри з кроком 20, val-кадри з кроком оцінки 10), бокси, щільність, частку малих об'єктів і розподіл класів у валідаційній частині кожного фолду.

In [ ]:
images["fold"] = images["clip"].map(common.clip_to_fold(splits))
obj["fold"] = obj.img_id.map(images.fold)
clip_stats["fold"] = clip_stats.index.map(common.clip_to_fold(splits))
fold_tab = images.groupby("fold").agg(clips=("clip", "nunique"), frames=("img_id", "size"),
                                      obj_per_frame=("n_obj", "mean"), ignore_area_pct=("ign_frac", "mean"))
fold_tab["ignore_area_pct"] *= 100
fold_tab["boxes"] = obj.groupby("fold").size()
fold_tab["small_%"] = (obj.size_cat == "small").groupby(obj.fold).mean().mul(100)
fold_tab["train_frames@stride20"] = [len(common.fold_images(images, splits, k, "train", common.TRAIN_STRIDE)) for k in fold_tab.index]
fold_tab["val_frames@stride10"] = [len(common.fold_images(images, splits, k, "val", common.EVAL_STRIDE)) for k in fold_tab.index]
fold_cls = pd.crosstab(obj.fold, obj.cname, normalize="index").mul(100).reindex(columns=common.CLASSES, fill_value=0)
fig, ax = plt.subplots(figsize=(9, 3.2))
fold_cls.T.plot.bar(ax=ax, logy=True, rot=0); ax.set_ylabel("share of boxes, % (log)"); ax.legend(title="val fold")
fig.savefig(IMG_OUT / "fold_balance.png"); plt.show()
display(fold_tab.round(2)); display(fold_cls.round(2))
pd.crosstab(clip_stats.fold, clip_stats.width.astype(str) + "x" + clip_stats.height.astype(str), rownames=["fold"], colnames=["clips by resolution"])

Самоперевірка оцінювання з `common.py` на val-частині фолду 0: GT як передбачення (очікуємо mAP ≈ 1), GT + хибні детекції з вищим скором всередині ігнору (метрика не має змінитися), GT + такі ж хибні детекції на фоні та зашумлені бокси з випадковими скорами (метрика має впасти).

In [ ]:
ids = common.fold_images(images, splits, 0, "val", common.EVAL_STRIDE).img_id.to_numpy()
gt_pred = obj.loc[obj.img_id.isin(ids), ["img_id", "cls", "x", "y", "w", "h"]].assign(score=0.9)
g = ign[ign.img_id.isin(ids)]
fp_ign = pd.DataFrame({"img_id": g.img_id, "cls": rng.integers(0, 8, len(g)), "x": g.x + g.w / 4, "y": g.y + g.h / 4,
                       "w": g.w / 2, "h": g.h / 2, "score": 0.99})
sz = images.loc[np.repeat(ids, 30), ["width", "height"]].to_numpy(float)
fp_bg = pd.DataFrame({"img_id": np.repeat(ids, 30), "cls": rng.integers(0, 8, len(sz)), "x": rng.random(len(sz)) * (sz[:, 0] - 24),
                      "y": rng.random(len(sz)) * (sz[:, 1] - 24), "w": 24.0, "h": 24.0, "score": 0.99})
jit = rng.normal(0, 0.15, (len(gt_pred), 2))
noisy = gt_pred.assign(x=gt_pred.x + jit[:, 0] * gt_pred.w, y=gt_pred.y + jit[:, 1] * gt_pred.h, score=rng.random(len(gt_pred)))
tests = {"GT as predictions": gt_pred, "GT + FP inside ignore": pd.concat([gt_pred, fp_ign]),
         "GT + FP on background": pd.concat([gt_pred, fp_bg]), "jittered GT, random scores": noisy}
t0 = time.time()
coco_gt = common.make_coco_gt(images, anns, ids)
ev = pd.DataFrame({k: common.evaluate(v, images, anns, ids, coco_gt=coco_gt) for k, v in tests.items()}).T
print(f"{len(ids)} frames | {(time.time() - t0) / len(tests):.1f}s per evaluation")
ev[["mAP", "AP50", "AP75", "AP_small", "AR500"]].round(4)

Перевіряємо агрегацію out-of-fold (по фолдах, mean ± std, pooled OOF) і запис у `results.csv` на штучних передбаченнях; тестовий запис пишемо в окремий файл, щоб не засмічувати журнал експериментів.

In [ ]:
fold_preds = {}
for k in range(common.N_FOLDS):
    ids_k = common.fold_images(images, splits, k, "val", common.EVAL_STRIDE).img_id.to_numpy()
    p = obj.loc[obj.img_id.isin(ids_k), ["img_id", "cls", "x", "y", "w", "h"]]
    fold_preds[k] = p.assign(score=rng.random(len(p)), x=p.x + rng.normal(0, 2, len(p)))
oof = common.evaluate_oof(fold_preds, images, anns, splits)
common.log_results(oof, path=WORK / "results_selftest.csv", owner="A", model="selftest", experiment="eval_selftest",
                   train_stride=common.TRAIN_STRIDE, eval_stride=common.EVAL_STRIDE, imgsz=common.IMGSZ)
display(oof[["mAP", "AP50", "AP_small", "AR500"]].round(4))
pd.read_csv(WORK / "results_selftest.csv").iloc[:, :12]

Перевіряємо запис сабміту: формуємо файл зі штучними боксами (зокрема такими, що виходять за межі кадру) і звіряємо формат та порядок рядків із `sample_submission.csv`.

In [ ]:
dummy = test_index.head(5)[["image_id"]].merge(
    pd.DataFrame({"cls": [2, 0], "score": [0.9, 0.5], "x": [10.0, -5.0], "y": [20.0, 30.0], "w": [30.0, 40.0], "h": [40.0, 4000.0]}), how="cross")
path = common.write_submission(dummy, test_index, WORK / "submission_selftest.csv")
print(common.check_submission(path, ROOT / "sample_submission.csv"))
pd.read_csv(path, keep_default_na=False).head(3)

## 6. Torchvision Dataset (Person A)
Створюємо датасети для фолду 0 з `tv_dataset.py`: train — кліпи train-частини з кроком 20 і горизонтальним фліпом, val — кліпи val-частини з кроком 10; ігнор заливається сірим 114, кадр масштабується до найдовшої сторони 640.

In [ ]:
from tv_dataset import VisDroneDet, make_loader, to_device, preds_to_df
f0 = splits["folds"][0]
ds_tr = VisDroneDet(images, anns, clips=f0["train_clips"], stride=common.TRAIN_STRIDE, max_side=common.IMGSZ, train=True)
ds_va = VisDroneDet(images, anns, clips=f0["val_clips"], stride=common.EVAL_STRIDE, max_side=common.IMGSZ, train=False)
print("train frames:", len(ds_tr), "| val frames:", len(ds_va))

Беремо один батч через DataLoader (2 воркери) і перевіряємо форму та тип тензорів, кількість боксів, мітки класів (1..8, 0 — фон у torchvision) і що всі бокси коректні та в межах кадру.

In [ ]:
dl = make_loader(ds_tr, batch_size=4, num_workers=2)
t0 = time.time()
imgs, tgts = next(iter(dl))
print(f"first batch: {time.time() - t0:.1f}s")
for im, t in zip(imgs, tgts):
    b = t["boxes"]
    ok = bool(((b[:, 2] > b[:, 0]) & (b[:, 3] > b[:, 1])).all() and (b >= 0).all()
              and (b[:, 2] <= im.shape[2]).all() and (b[:, 3] <= im.shape[1]).all())
    print(tuple(im.shape), im.dtype, "| boxes", tuple(b.shape), "| labels", sorted(t["labels"].unique().tolist()),
          "| scale", round(t["scale"], 3), "| orig", t["orig_size"], "| valid:", ok)
x, y = to_device(imgs, tgts, "cpu")
print("after to_device:", x[0].dtype, round(float(x[0].min()), 3), round(float(x[0].max()), 3))

Вимірюємо швидкість завантаження 20 батчів на CPU, щоб оцінити, чи не стане DataLoader вузьким місцем під час навчання на GPU.

In [ ]:
t0, n = time.time(), 0
for i, (imgs, tgts) in enumerate(dl):
    n += len(imgs)
    if i == 19:
        break
print(f"{n / (time.time() - t0):.1f} images/s with {dl.num_workers} workers")

Візуалізуємо останній батч так, як його бачить модель: масштабовані кадри з залитим ігнором і бокси після фліпу.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 3.6))
for ax, im, t in zip(axes, imgs, tgts):
    ax.imshow(im.permute(1, 2, 0).numpy())
    for (x1, y1, x2, y2), l in zip(t["boxes"].tolist(), t["labels"].tolist()):
        ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, ec=COLORS[l - 1], lw=0.7))
    ax.set_title(f"img_id {t['image_id']} | {len(t['boxes'])} boxes", fontsize=8); ax.axis("off")
fig.savefig(IMG_OUT / "batch_sanity.png"); plt.show()

Перевіряємо зворотне перетворення координат для майбутніх передбачень: цілі з val-датасету подаємо як «передбачення», `preds_to_df` переводить їх в оригінальні пікселі, а `common.evaluate` рахує mAP (має бути близьким до 1; відхилення — лише через відкинуті бокси під ігнором).

In [ ]:
outs, tg = [], []
for i in np.linspace(0, len(ds_va) - 1, min(60, len(ds_va))).astype(int):
    _, t = ds_va[i]
    outs.append({"boxes": t["boxes"], "labels": t["labels"], "scores": torch.ones(len(t["labels"]))})
    tg.append(t)
rt = preds_to_df(outs, tg)
r = common.evaluate(rt, images, anns, [t["image_id"] for t in tg])
print({k: round(r[k], 4) for k in ["mAP", "AP50", "AP_small"]})

## 7. Статистика боксів і k-means якорі по фолдах (Person A)
Для кожного фолду беремо лише бокси train-частини (без перекритих ігнором), переводимо їх у масштаб входу моделі (найдовша сторона 640) і рахуємо квантилі розміру та співвідношення сторін h/w.

In [ ]:
sc = common.IMGSZ / np.maximum(obj.width, obj.height)
obj["w640"], obj["h640"] = obj.w * sc, obj.h * sc
obj["ar"] = obj.h / np.maximum(obj.w, 1e-3)
q = [0.05, 0.25, 0.5, 0.75, 0.95]
stats = {}
for f in splits["folds"]:
    o = obj[obj["clip"].isin(f["train_clips"]) & (obj.ign_cover < 0.5)]
    stats[(f["fold"], "sqrt_area@640")] = np.sqrt(o.w640 * o.h640).quantile(q).values
    stats[(f["fold"], "h/w")] = o.ar.quantile(q).values
display(pd.DataFrame(stats, index=[f"q{int(v * 100)}" for v in q]).T.round(2))
obj.groupby("cname").ar.median().reindex(common.CLASSES).round(2).rename("median h/w").to_frame().T

Реалізуємо k-means з відстанню 1 − IoU (як у YOLOv2), 1D k-means у лог-шкалі та генерацію сітки якорів, яка повторює torchvision `AnchorGenerator` для рівнів P3–P7 (кроки 8–128) на вході 640 px.

In [ ]:
def wh_iou(wh, c):
    inter = np.minimum(wh[:, None, 0], c[None, :, 0]) * np.minimum(wh[:, None, 1], c[None, :, 1])
    return inter / (wh.prod(1)[:, None] + c.prod(1)[None] - inter)

def kmeans_iou(wh, k, iters=300, seed=common.SEED):
    c = wh[np.random.default_rng(seed).choice(len(wh), k, replace=False)]
    for _ in range(iters):
        a = wh_iou(wh, c).argmax(1)
        new = np.array([np.median(wh[a == j], 0) if (a == j).any() else c[j] for j in range(k)])
        if np.allclose(new, c):
            break
        c = new
    c = c[np.argsort(c.prod(1))]
    return c, float(wh_iou(wh, c).max(1).mean())

def kmeans_1d(v, k, iters=100):
    c = np.quantile(v, (np.arange(k) + 0.5) / k)
    for _ in range(iters):
        a = np.abs(v[:, None] - c[None]).argmin(1)
        new = np.array([v[a == j].mean() if (a == j).any() else c[j] for j in range(k)])
        if np.allclose(new, c):
            break
        c = new
    return np.sort(c)

def tv_sizes(bases):
    return tuple(tuple(max(1, int(b * 2 ** (i / 3))) for i in range(3)) for b in bases)

def grid_anchors(h, w, sizes, ratios, strides=(8, 16, 32, 64, 128)):
    out = []
    for s, sz, rt in zip(strides, sizes, ratios):
        hr = np.sqrt(np.asarray(rt, float)); wr = 1 / hr
        ws = (wr[:, None] * np.asarray(sz, float)[None]).ravel(); hs = (hr[:, None] * np.asarray(sz, float)[None]).ravel()
        base = np.round(np.stack([-ws, -hs, ws, hs], 1) / 2)
        sx, sy = np.meshgrid(np.arange(math.ceil(w / s)) * s, np.arange(math.ceil(h / s)) * s)
        shifts = np.stack([sx.ravel(), sy.ravel(), sx.ravel(), sy.ravel()], 1)
        out.append((shifts[:, None] + base[None]).reshape(-1, 4))
    return torch.as_tensor(np.concatenate(out), dtype=torch.float32)

def best_anchor_iou(boxes, shapes, sizes, ratios):
    from torchvision.ops import box_iou
    cache, best = {}, []
    for b, (h, w) in zip(boxes, shapes):
        if len(b) == 0:
            continue
        key = (math.ceil(h / 32) * 32, math.ceil(w / 32) * 32)
        if key not in cache:
            cache[key] = grid_anchors(*key, sizes, ratios)
        best.append(box_iou(torch.as_tensor(b, dtype=torch.float32), cache[key]).max(1).values.numpy())
    return np.concatenate(best) if best else np.zeros(0)

Для кожного фолду на train-боксах рахуємо k-means (k = 9) по (w, h) і конфігурацію для torchvision RetinaNet (5 рівнів × 3 розміри × 3 співвідношення: розміри з 1D k-means по √площі, співвідношення з 1D k-means по h/w). Потім на 300 val-кадрах фолду рахуємо, яка частка GT має якір з IoU ≥ 0.5 / 0.4 для стандартних, зменшених у 4 рази і k-means якорів.

In [ ]:
mk = lambda bases, ratios=(0.5, 1.0, 2.0): dict(sizes=tv_sizes(bases), ratios=(tuple(ratios),) * 5)
CONFIGS = {"default": mk([32, 64, 128, 256, 512]), "small_x0.25": mk([8, 16, 32, 64, 128])}
ocover, osize = obj.ign_cover.to_numpy(), obj.size_cat.to_numpy()
anchor_cfg, cov_rows = {}, []
t0 = time.time()
for f in splits["folds"]:
    k = f["fold"]
    o = obj[obj["clip"].isin(f["train_clips"]) & (obj.ign_cover < 0.5)]
    o = o.sample(min(len(o), 200_000), random_state=common.SEED)
    wh = o[["w640", "h640"]].to_numpy(np.float64)
    wh = wh[(wh > 0).all(1)]
    c9, miou = kmeans_iou(wh, 9)
    bases = np.exp(kmeans_1d(np.log(np.sqrt(wh.prod(1))), 5)) / 2 ** (1 / 3)
    ratios = np.round(np.exp(kmeans_1d(np.log(wh[:, 1] / wh[:, 0]), 3)), 2)
    cfgs = {**CONFIGS, "kmeans": mk(bases, ratios.tolist())}
    va = common.fold_images(images, splits, k, "val", common.TRAIN_STRIDE)
    va = va.sample(min(len(va), 300), random_state=common.SEED)
    boxes, shapes, cats = [], [], []
    for iid, r in va.iterrows():
        s = common.IMGSZ / max(r.width, r.height)
        a, b = np.searchsorted(oid, [iid, iid + 1])
        sel = ocover[a:b] < 0.5
        bb = oxywh[a:b][sel] * s
        boxes.append(np.column_stack([bb[:, 0], bb[:, 1], bb[:, 0] + bb[:, 2], bb[:, 1] + bb[:, 3]]))
        shapes.append((round(r.height * s), round(r.width * s)))
        cats.append(osize[a:b][sel])
    cats = np.concatenate(cats)
    for name, cfg in cfgs.items():
        best = best_anchor_iou(boxes, shapes, **cfg)
        row = {"fold": k, "anchors": name, "IoU>=0.5 %": 100 * (best >= 0.5).mean(),
               "IoU>=0.4 %": 100 * (best >= 0.4).mean(), "mean best IoU": best.mean()}
        row.update({f"{c} IoU>=0.5 %": 100 * (best[cats == c] >= 0.5).mean() if (cats == c).any() else np.nan for c in ["small", "medium", "large"]})
        cov_rows.append(row)
    anchor_cfg[k] = {"kmeans_wh_k9@640": np.round(c9, 2).tolist(), "kmeans_k9_mean_iou": round(miou, 4),
                     "retinanet": {n: {"sizes": c["sizes"], "aspect_ratios": c["ratios"]} for n, c in cfgs.items()}}
cov = pd.DataFrame(cov_rows)
print(f"{time.time() - t0:.0f}s")
display(pd.DataFrame({k: {"k9 mean IoU": v["kmeans_k9_mean_iou"], "kmeans sizes": v["retinanet"]["kmeans"]["sizes"],
                          "kmeans ratios": v["retinanet"]["kmeans"]["aspect_ratios"][0]} for k, v in anchor_cfg.items()}).T)
cov.groupby("anchors", sort=False).agg(["mean", "std"]).drop(columns="fold").round(1)

Візуалізуємо k-means центроїди фолду 0 поверх розподілу (w, h) train-боксів і зберігаємо всі конфігурації якорів у `anchors.json` для ноутбука RetinaNet.

In [ ]:
o = obj[obj["clip"].isin(splits["folds"][0]["train_clips"]) & (obj.ign_cover < 0.5)]
o = o.sample(min(len(o), 30_000), random_state=common.SEED)
c = np.array(anchor_cfg[0]["kmeans_wh_k9@640"])
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(o.w640, o.h640, s=1, alpha=0.15, label="train boxes, fold 0")
ax.scatter(c[:, 0], c[:, 1], c="red", marker="x", s=70, label="k-means, k=9")
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("w @640, px"); ax.set_ylabel("h @640, px"); ax.legend()
fig.savefig(IMG_OUT / "anchors_kmeans.png"); plt.show()
with open(WORK / "anchors.json", "w") as fh:
    json.dump({str(k): v for k, v in anchor_cfg.items()}, fh, indent=1)

## 8. Збереження артефактів
Зберігаємо таблицю статистик по кліпах і виводимо список файлів, які потрібно перенести в репозиторій.

In [ ]:
clip_stats.round(3).to_csv(WORK / "clip_stats.csv")
for p in sorted([*WORK.glob("*.json"), *WORK.glob("*.csv"), *IMG_OUT.glob("*.png")]):
    print(f"{str(p):<60} {p.stat().st_size / 1e3:8.0f} KB")